# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task type: ranking or scoring with a supervised check and not pure classification

The decision is which pages are to be reviewed first, not a yes or no answer on a single page in isolation. The output is a priority score + reason codes (`ctr_gap`, `engagement_gap`, `both`) that sorts a review queue. 
I will also define an observed binary label in a later time window to validate whether top‑K ranked pages were real opportunities — that will use precision/recall, but the product artifact is the ranked list.

Why not the other types:
- Classification alone hides ordering but reviewers need a queue not just a flag.
- Clustering finds archetypes but not the exact snippets to fix first.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

Score: how strong the CTR or engagement opportunity looks right now (tier-adjusted CTR gap + weak engagement given sessions).

Label (for validation): did the opportunity show up in a later time window? This is measured from raw clicks, impressions, and sessions. Not from `trend_direction` / `trend_pct`, and not from a hand-written rule.

- CTR arm: page underperforms its position tier on CTR, with enough impressions.
- Engagement arm: enough sessions but low engagement rate.

In the warehouse, labels come from daily facts with separate feature and label windows. On the starter CSV, the code below uses prev-30d → last-30d as a small preview of that idea.



In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path("../../data/raw/content_refresh_anonymized.csv"))


ctr = df[(df["avg_position"] > 0) & (df["impressions_prev_30d"] >= 200)].copy()
ctr["ctr_prev30"] = 100 * ctr["clicks_prev_30d"] / ctr["impressions_prev_30d"]
ctr["ctr_last30"] = 100 * ctr["clicks_last_30d"] / ctr["impressions_last_30d"]
ctr["tier_med_prev"] = ctr.groupby("position_tier")["ctr_prev30"].transform("median")
ctr["tier_med_last"] = ctr.groupby("position_tier")["ctr_last30"].transform("median")
ctr["gap_prev_pp"] = ctr["tier_med_prev"] - ctr["ctr_prev30"]
ctr["gap_last_pp"] = ctr["tier_med_last"] - ctr["ctr_last30"]
ctr["label_persistent_ctr_gap"] = (
    (ctr["gap_prev_pp"] > 0.1)
    & (ctr["gap_last_pp"] > 0.1)
    & (ctr["impressions_last_30d"] >= 200)
)

print("Starter preview — observed CTR opportunity label")
print(f"  Rows with prev-30d volume floor: {len(ctr):,}")
print(
    f"  Persistent tier-underperformance (prev + last 30d): "
    f"{ctr['label_persistent_ctr_gap'].mean():.1%}"
)
print("  (Warehouse will replace this with daily-fact windows + GA4 engagement arm.)")


Starter preview — observed CTR opportunity label
  Rows with prev-30d volume floor: 15,212
  Persistent tier-underperformance (prev + last 30d): 11.0%
  (Warehouse will replace this with daily-fact windows + GA4 engagement arm.)


## 3. Success metric

precision@50 — of the top 50 pages my score ranks highest, how many were real opportunities?

Good means beating the simple tier-gap baseline. On the starter preview below, that baseline hits ~40% precision@50 vs ~11% base rate (~3.6× lift). My model should do better than that on a client holdout split.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50

baseline_score = ctr["gap_prev_pp"].clip(lower=0)
labels = ctr["label_persistent_ctr_gap"].astype(int)


def precision_at_k(score: pd.Series, y: pd.Series, k: int) -> float:
    top_idx = score.nlargest(k).index
    return y.loc[top_idx].mean()


p_at_k = precision_at_k(baseline_score, labels, K)
base_rate = labels.mean()
lift = p_at_k / base_rate if base_rate else float("nan")

print(f"Primary metric demo — precision@{K} (CTR arm, starter preview)")
print(f"  Base rate (persistent gap): {base_rate:.1%}")
print(f"  Tier-gap baseline precision@{K}: {p_at_k:.1%}")
print(f"  Lift vs base rate: {lift:.2f}x")
print(f"  'Good' later = beat {p_at_k:.1%} with a richer score + stable on client holdout")


Primary metric demo — precision@50 (CTR arm, starter preview)
  Base rate (persistent gap): 11.0%
  Tier-gap baseline precision@50: 40.0%
  Lift vs base rate: 3.63x
  'Good' later = beat 40.0% with a richer score + stable on client holdout


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (`content_id`) for one client at one snapshot in time.

The code below loads my lane slice and adds a score plus a reason code for each page.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane = df[
    (df["avg_position"] > 0) | (df["sessions_90d"] >= 50)
].copy()

# Tier medians from the CTR-eligible population (position known + volume floor)
ctr_ref = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)]
tier_med_ctr = ctr_ref.groupby("position_tier")["ctr"].median()
lane["tier_med_ctr"] = lane["position_tier"].map(tier_med_ctr)
lane["ctr_gap_pp"] = lane["tier_med_ctr"] - lane["ctr"]
lane["ctr_flag"] = (
    (lane["avg_position"] > 0)
    & (lane["impressions_90d"] >= 500)
    & (lane["ctr_gap_pp"] > 0.1)
)

# Engagement arm
lane["eng_flag"] = (lane["sessions_90d"] >= 50) & (lane["engagement_rate"] < 30)

lane["reason_code"] = np.select(
    [lane["ctr_flag"] & lane["eng_flag"], lane["ctr_flag"], lane["eng_flag"]],
    ["both", "ctr_gap", "engagement_gap"],
    default="none",
)

# Simple baseline score: exposure × strongest normalized gap
lane["ctr_score"] = (
    lane["ctr_gap_pp"].clip(lower=0)
    * np.log1p(lane["impressions_90d"].fillna(0))
)
lane["eng_score"] = (
    ((30 - lane["engagement_rate"].clip(upper=30)) / 30)
    * np.log1p(lane["sessions_90d"].fillna(0))
)
lane["opportunity_score"] = lane[["ctr_score", "eng_score"]].max(axis=1)

show_cols = [
    "content_id",
    "client_id",
    "position_tier",
    "impressions_90d",
    "ctr",
    "ctr_gap_pp",
    "sessions_90d",
    "engagement_rate",
    "reason_code",
    "opportunity_score",
]

print(f"Unit of analysis: one row = one content item")
print(f"Lane slice shape: {lane.shape[0]:,} rows x {lane.shape[1]} columns")
print(f"Reason codes:\n{lane['reason_code'].value_counts().to_string()}\n")
lane.sort_values("opportunity_score", ascending=False)[show_cols].head(5)


Unit of analysis: one row = one content item
Lane slice shape: 28,798 rows x 52 columns
Reason codes:
reason_code
none              21166
engagement_gap     4492
ctr_gap            2713
both                427



,content_id,client_id,position_tier,impressions_90d,ctr,ctr_gap_pp,sessions_90d,engagement_rate,reason_code,opportunity_score
9678,content_4560b0a818ab,client_19581e27de,page_1,4238,0.21,0.03,4345,0.05,engagement_gap,8.363049
14234,content_89e84d699e9e,client_349c41201b,page_1,275226,0.89,-0.65,2554,0.70,engagement_gap,7.662739
29400,content_2dba2b1f9536,client_6208ef0f77,page_3_5,443434,0.21,-0.12,4218,2.73,engagement_gap,7.587744
16811,content_8e7ba84a972b,client_7f2253d7e2,page_1,288426,0.92,-0.68,2598,1.19,engagement_gap,7.550988
14090,content_44e481c8f55b,client_19581e27de,top_3,312694,0.65,-0.45,1963,0.76,engagement_gap,7.390642


## 5. Why ML beats a fixed rule here

A simple rule like "below tier CTR or engagement < 30%" is a fine baseline, but one cutoff does not fit every page. CTR gaps differ a lot by content type (~81% for comparison articles vs ~25% for keyword articles). Engagement gaps differ by intent. Only ~427 pages hit both signals, so one if-statement cannot rank the full queue well. The code below shows those splits.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evidence: gap prevalence is not uniform -> one rule fits poorly
ctr_eligible = df[
    (df["avg_position"] > 0) & (df["impressions_90d"] >= 500)
].copy()
ctr_eligible["ctr_gap_pp"] = ctr_eligible.groupby("position_tier")[
    "ctr"
].transform(lambda s: s.median() - s)

gap_by_type = (
    ctr_eligible.assign(flag=ctr_eligible["ctr_gap_pp"] > 0.1)
    .groupby("content_type")["flag"]
    .mean()
    .sort_values(ascending=False)
)

eng_by_intent = (
    df.assign(
        eng_flag=(df["sessions_90d"] >= 50) & (df["engagement_rate"] < 30)
    )
    .groupby("main_intent", dropna=False)["eng_flag"]
    .mean()
    .sort_values(ascending=False)
)

print("CTR gap rate (>0.1pp below tier) by content_type:")
print((100 * gap_by_type).round(1).astype(str) + "%")

print("\nEngagement gap rate by main_intent (top 4):")
print((100 * eng_by_intent.head(4)).round(1).astype(str) + "%")

both_count = len(
    df[
        (df["avg_position"] > 0)
        & (df["impressions_90d"] >= 500)
        & (df["sessions_90d"] >= 50)
        & (df["engagement_rate"] < 30)
    ].merge(
        ctr_eligible.loc[ctr_eligible["ctr_gap_pp"] > 0.1, ["content_id"]],
        on="content_id",
    )
)
print(
    f"\nOverlap: both flags on same page = {both_count:,} pages — "
    "a single if-statement cannot rank these jointly."
)


CTR gap rate (>0.1pp below tier) by content_type:
content_type
comparison article    67.1%
feedly article        25.0%
keyword article       18.5%
Name: flag, dtype: str

Engagement gap rate by main_intent (top 4):
main_intent
commercial       19.6%
informational    17.6%
transactional    16.1%
navigational      8.7%
Name: eng_flag, dtype: str

Overlap: both flags on same page = 427 pages — a single if-statement cannot rank these jointly.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.